# Lab 05: Column Generation

**Author:** Bo Tang

## Setup

See the [README](README.md#getting-started) for installation. Skip the `%pip` cell if `gurobipy` and `numpy` are already installed in this kernel.

In [ ]:
# Optional: run only if these packages are missing from this kernel.
%pip install gurobipy numpy

In [1]:
import gurobipy as gp
import numpy as np
from gurobipy import GRB

## Column Generation

In cutting stock, each variable counts how often a cutting pattern is used. A column lists the quantities of each item produced by that pattern. There may be too many patterns to enumerate, even though an optimal plan uses only a few.

Column generation alternates between choosing quantities for the patterns already available and searching for another useful pattern. For the cost-minimization convention used in the first example, the full LP is

$$
\min_{x\geq0}\ c^\top x\quad\text{subject to}\quad Ax\geq b,
$$

where $b$ is demand, column $a_j$ describes pattern $j$, and $c_j$ is its cost.

### Restricted Master Problem

For the current pattern set $J$, solve

$$
\min_{x\geq0}\sum_{j\in J}c_jx_j
\quad\text{subject to}\quad\sum_{j\in J}a_jx_j\geq b.
$$

Choose initial patterns that can meet all demands. This LP selects their quantities and provides a dual price $\lambda_i$ for each demand constraint. Adding patterns gives the minimization model more choices, so its objective cannot increase.

### Pricing Problem

The dual prices value a pattern's output at $\lambda^\top a_j$. Subtract that value from the pattern's cost to obtain its reduced cost:

$$
\bar c_j=c_j-\lambda^\top a_j.
$$

A negative reduced cost identifies a candidate that can improve this minimization master. The pricing problem searches over feasible patterns for such a column, rather than listing every pattern in advance. Add it, solve the master again, and update the prices.

If exact pricing finds no negative reduced cost within tolerance, the optimal restricted master also solves the full LP. For the maximization shipment example later in the lab, the improving sign is positive instead.

See the [MIT column-generation notes](https://ocw.mit.edu/courses/1-224j-carrier-systems-fall-2003/132da550d77666a74f5417e3f7dc4d1a_lecture10.pdf) for a worked formulation.

### Integer Variables

Column generation solves the LP relaxation. Making the generated variables integer afterward gives a restricted integer model, which may omit patterns needed for the global integer optimum.

**Branch-and-price** addresses this by using column generation inside branch-and-bound, with pricing that respects the branching decisions.

## Example 1: Cutting Stock

Cut 107-foot copper-wire stock into 3-, 5-, and 9-foot pieces to meet demands of 25, 20, and 15 pieces, respectively. Minimize the number of stock lengths used.

In [2]:
# Available lengths
lengths = [3, 5, 9]
# Demand
demand = [25, 20, 15]

### Full Integer Model

Let $a_{ij}$ count pieces of length $i$ in pattern $j$, and let $x_j$ count how often that pattern is used. Enumerating all $k$ feasible patterns gives

$$
\min_x \sum_{j=1}^k x_j
$$
Subject to
$$
\sum_{j=1}^k a_{3j} x_j \geq 25
$$
$$
\sum_{j=1}^k a_{5j} x_j \geq 20
$$
$$
\sum_{j=1}^k a_{9j} x_j \geq 15
$$
$$
x_j \in \mathbb{Z}_{\geq 0} \quad \forall j
$$

**Discussion questions:**

- How many cutting patterns do we have?

In [3]:
import numpy as np

# Wholesale length of copper wire
wholesale_length = 107

# Generate all combinations of cutting patterns
def generate_combinations():
    combinations = []
    for a3 in range(wholesale_length // lengths[0] + 1):
        for a5 in range(wholesale_length // lengths[1] + 1):
            for a9 in range(wholesale_length // lengths[2] + 1):
                if (a3 * lengths[0] + a5 * lengths[1] + a9 * lengths[2]) <= wholesale_length:
                    combinations.append((a3, a5, a9))
    return combinations

# Generate all possible cutting patterns
patterns = generate_combinations()

# Display the number of patterns and a sample
print("Total number of valid cutting patterns:", len(patterns))
print("Sample patterns:", patterns[:10])  # Show first 10 patterns

Total number of valid cutting patterns: 1921
Sample patterns: [(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 0, 3), (0, 0, 4), (0, 0, 5), (0, 0, 6), (0, 0, 7), (0, 0, 8), (0, 0, 9)]


**Result:** The enumeration contains 1,921 patterns, including the empty pattern $(0,0,0)$. The empty pattern cannot help meet demand and costs one stock length, so it will not be used in an optimal solution.

In [4]:
# Create Gurobi model
m = gp.Model("cutting_stock")

# Decision variables, one for each cutting pattern
x = m.addVars(len(patterns), vtype=GRB.INTEGER, name="x")

# Objective: Minimize the total number of wires used
m.setObjective(gp.quicksum(x[i] for i in range(len(patterns))), GRB.MINIMIZE)

# Demand constraints for each length 3 5 9
m.addConstr(gp.quicksum(patterns[i][0] * x[i] for i in range(len(patterns))) >= demand[0], "Demand_3ft")
m.addConstr(gp.quicksum(patterns[i][1] * x[i] for i in range(len(patterns))) >= demand[1], "Demand_5ft")
m.addConstr(gp.quicksum(patterns[i][2] * x[i] for i in range(len(patterns))) >= demand[2], "Demand_9ft")

# Optimize the model
m.optimize()

# Display the results
if m.status == GRB.OPTIMAL:
    print("\nOptimal solution found:")
    print(f"Minimum number of wires needed: {m.objVal}\n")
    print("Cutting patterns used:")
    for i in range(len(patterns)):
        if x[i].x > 0:
            print(f"Pattern {patterns[i]}: used {x[i].x} times")
else:
    print("No optimal solution found.")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26300.2))


Optimize a model with 3 rows, 1921 columns and 4970 nonzeros (Min)
Model fingerprint: 0x21c94254
Model has 1921 linear objective coefficients
Variable types: 0 continuous, 1921 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+01, 3e+01]

Found heuristic solution: objective 35.0000000
Presolve removed 0 rows and 68 columns
Presolve time: 0.00s
Presolved: 3 rows, 1853 columns, 4830 nonzeros
Found heuristic solution: objective 3.0000000
Variable types: 0 continuous, 1853 integer (2 binary)

Root relaxation: cutoff, 4 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0     cutoff    0         3.00000    3.00000  0.00%     -    0s

Explo

**Result:** Three stock lengths suffice. Demand requires $25(3)+20(5)+15(9)=310$ feet, so two 107-foot lengths cannot suffice. This proves that the displayed three-length plan is optimal, even if another solver run selects different patterns.

## Column Generation for Cutting Stock

The restricted master chooses how often to use each available pattern. The pricing problem generates a new pattern using the master LP's dual prices. Keep the master variables continuous while generating columns.

### Initial Pattern

Start with $(1,1,11)$: one 3-foot piece, one 5-foot piece, and eleven 9-foot pieces. This uses all 107 feet.

In [5]:
init_pattern = [1, 1, 11]

### Exercise: Restricted Master Problem

Complete the continuous nonnegative pattern variables and demand constraints. Keep the constraint names used by `getConstrByName`.

Return the objective and three dual prices in length order $(3,5,9)$. With only the initial pattern, check that the model uses 25 stock lengths and meets every demand.

In [ ]:
def solve_master_problem(patterns):
    # Create Gurobi model
    m = gp.Model("master")
    # Turn off log
    m.Params.outputFlag = 0
    # TODO: decision variables: nonnegative continuous pattern counts
    x = m.addVars
    # Objective: Minimize the number of 107' copper wires used
    m.setObjective(gp.quicksum(x[i] for i in range(len(patterns))), GRB.MINIMIZE)
    # TODO: demand constraints (>=), named Demand_3ft, Demand_5ft, Demand_9ft
    m.addConstr
    # Optimize the master
    m.optimize()
    if m.Status != GRB.OPTIMAL:
        raise RuntimeError("The column-generation model was not solved to optimality.")
    # Objective value
    objval = m.objVal
    # Return the dual variables (shadow prices) from the demand constraints
    λ = [m.getConstrByName("Demand_3ft").Pi,
         m.getConstrByName("Demand_5ft").Pi,
         m.getConstrByName("Demand_9ft").Pi]
    return objval, λ

# Solve master problem
objval, λ = solve_master_problem([init_pattern])

### Exercise: Pricing Problem

For a pattern $a=(a_3,a_5,a_9)$, the reduced cost is $1-\lambda^\top a$. Complete the integer knapsack model:

$$
\max\ \lambda^\top a-1
\quad\text{subject to}\quad
3a_3+5a_5+9a_9\leq107,\qquad a\in\mathbb{Z}_{\geq0}^3.
$$

Return the pattern and improvement score $\lambda^\top a-1$. The loop calls this `improvement`; it is the negative of reduced cost.

Check that the returned pattern has integer nonnegative entries, uses at most 107 feet, and has the reported improvement score.

In [ ]:
def solve_subproblem(λ):
    # Create the subproblem to find new cutting patterns
    m = gp.Model("Subproblem")
    # Turn off log
    m.Params.outputFlag = 0
    # TODO: decision variables: nonnegative integer counts of 3', 5', and 9' pieces

    # TODO: objective: maximize the improvement score (dual-weighted pattern value minus 1)

    # TODO: constraint: total cut length <= 107

    # Optimize the subproblem
    m.optimize()
    if m.Status != GRB.OPTIMAL:
        raise RuntimeError("The column-generation model was not solved to optimality.")
    # Objective value
    objval = m.objVal
    # TODO: get new pattern in the order [3ft, 5ft, 9ft]
    new_pattern =
    return objval, new_pattern

objval, new_pattern = solve_subproblem(λ)
print("\nObjective value:", objval)
print("New pattern:", new_pattern)

### Exercise: Column Generation Loop

Run the master and pricing problems until no improvement score exceeds $10^{-6}$. The loop below applies this tolerance.

Report the master objective, new pattern, and improvement score at each iteration. The minimization master objective should not increase as columns are added. Save its final LP objective before solving the integer model.

In [ ]:
# Exercise: complete the master and pricing TODOs before running this cell.
def column_generation(init_pattern, tol=1e-6, max_iterations=1000):
    patterns = [init_pattern]
    for iteration in range(1, max_iterations + 1):
        objval, λ = solve_master_problem(patterns)
        improvement, new_pattern = solve_subproblem(λ)
        print(f"Iteration {iteration}: LP objective = {objval:.6f}, improvement = {improvement:.6g}")
        if improvement <= tol:
            print(f"Final LP objective: {objval:.6f}")
            return patterns
        patterns.append(new_pattern)
        print(f"  Added pattern: {new_pattern}")
    raise RuntimeError("Column-generation iteration limit reached before convergence.")

patterns = column_generation(init_pattern)
print("Number of generated patterns:", len(patterns))
print("Patterns:", patterns)

### Exercise: Integer Solution over Generated Patterns

Solve the integer master using the generated patterns. Report the patterns used and the number of stock lengths. Check that the plan meets demand, then compare its objective with both the column-generation LP bound and the three-length enumerated optimum.

In [ ]:
# Exercise: complete the preceding TODOs before running this cell.
# Create Gurobi model
m = gp.Model("cutting_stock")

# Decision variables, one for each cutting pattern
x = m.addVars(len(patterns), vtype=GRB.INTEGER, name="x")

# Objective: Minimize the total number of wires used
m.setObjective(gp.quicksum(x[i] for i in range(len(patterns))), GRB.MINIMIZE)

# Demand constraints for each length 3 5 9
m.addConstr(gp.quicksum(patterns[i][0] * x[i] for i in range(len(patterns))) >= demand[0], "Demand_3ft")
m.addConstr(gp.quicksum(patterns[i][1] * x[i] for i in range(len(patterns))) >= demand[1], "Demand_5ft")
m.addConstr(gp.quicksum(patterns[i][2] * x[i] for i in range(len(patterns))) >= demand[2], "Demand_9ft")

# Optimize the model
m.optimize()

# Display the results
print("\nOptimal solution found:")
print(f"Minimum number of wires needed: {m.objVal}\n")
print("Cutting patterns used:")
for i in range(len(patterns)):
    if x[i].x > 0:
        print(f"Pattern {patterns[i]}: used {x[i].x} times")

If the generated-pattern integer solution uses more than three stock lengths, explain why solving the master to integer optimality does not establish optimality over patterns that were never generated.

## Example 2: Clothing Shipments

For each item $i\in F$, revenue is $r_i$, handling cost is $c_i$, and available quantity is $d_i$. Select at most $K$ distinct shipment patterns to maximize revenue. A feasible pattern's handling cost cannot exceed 50% of its revenue.

Let $S$ contain all feasible patterns, $a_{is}$ be the number of units of item $i$ in pattern $s$, and $r_s=\sum_i a_{is}r_i$. Set $x_s=1$ when selecting pattern $s$:

$$
\begin{aligned}
\max\quad & \sum_{s\in S}r_sx_s\\
\text{subject to}\quad & \sum_{s\in S}a_{is}x_s\leq d_i &&\forall i\in F,\\
& \sum_{s\in S}x_s\leq K,\\
& x_s\in\{0,1\} &&\forall s\in S.
\end{aligned}
$$

In the LP relaxation, denote the inventory-constraint dual prices by $\alpha_i$ and the shipment-count dual price by $\beta$.

**Exercise:** Write the reduced cost of a missing shipment column and formulate the pricing problem. Specify the item-count variables, inventory bounds, and handling-cost restriction. State the sign of reduced cost that improves this maximization master. Account for the LP relaxation's upper bound $x_s\leq1$ when discussing columns already in the master.